# 🧬 Trabalho de IA — Caixeiro Viajante com Algoritmo Genético

Neste trabalho, será desenvolvido um Algoritmo Genético de forma gradual, começando pela construção de uma população inicial e avançando para a implementação dos principais operadores evolutivos.
Ao longo das etapas, serão analisados o funcionamento do algoritmo, a qualidade das soluções encontradas e a evolução da população.


## Objetivos de aprendizagem

- representar uma rota como **cromossomo** (uma permutação);
- calcular a aptidão (custo) de uma rota usando a matriz de distâncias;
- entender seleção (roleta e torneio), cruzamento **OX** e mutação **swap**;
- acompanhar qualidade e diversidade ao longo das gerações;



## 1. Problema do Caixeiro Viajante

Há 6 cidades, numeradas de 0 a 5. O caixeiro **sai da cidade 0**, visita cada uma das cidades 1, 2, 3, 4 e 5 **exatamente uma vez** e **retorna à cidade 0**. O objetivo é **minimizar** a distância total percorrida.

### Tamanho do espaço de busca

Como a cidade 0 é fixa no início e no fim, só precisamos ordenar as 5 cidades intermediárias:

$$
5! = 120 \text{ rotas possíveis.}
$$


## 2. Modelagem do Algoritmo Genético

| Conceito | No trabalho|
|---|---|
| Gene | Uma cidade intermediária |
| Cromossomo | Lista com as cidades `[1..5]` em alguma ordem, por exemplo `[3, 5, 1, 4, 2]` |
| Indivíduo | Uma rota candidata (a rota completa é `[0] + cromossomo + [0]`) |
| População | Lista de indivíduos |
| Aptidão | Distância total da rota |
| Objetivo | **Minimizar** a distância |
| Seleção | Roleta (com peso $1/\text{distância}$) e Torneio |
| Cruzamento | OX (*Order Crossover*) |
| Mutação | Swap (troca de duas cidades) |
| Elitismo | Cópia do melhor indivíduo para a próxima geração |

O ciclo utilizado será:

$$
\boxed{\text{população}}
\rightarrow
\boxed{\text{avaliação}}
\rightarrow
\boxed{\text{seleção}}
\rightarrow
\boxed{\text{cruzamento e mutação}}
\rightarrow
\boxed{\text{nova população}}.
$$

## 🧭 Como ler este notebook

- Rode as células **em ordem**: cada uma depende das anteriores.
- Antes de cada célula de código há uma caixa **🔎 Entendendo o código** com: o que o código faz, o passo a passo e como interpretar a saída.
- As partes do algoritmo são construídas aos poucos: primeiro **uma geração à mão** (Parte I) e depois o **AG completo** (Parte II), validado com experimentos.


## 🗺️ Mapa das funções

| Função | Etapa | Para que serve |
|---|---|---|
| `gerar_matriz_distancias` | 2 | cria a matriz de distâncias simétrica |
| `gerar_individuo` / `gerar_populacao` | 3 | criam rotas aleatórias válidas |
| `rota_completa` | 3 | coloca a cidade 0 no início e no fim |
| `calcular_fitness` | 4 | distância total da rota (menor = melhor) |
| `calcular_pesos_roleta` / `calcular_probabilidades` | 3 (Parte I) | convertem custo em chance de seleção |
| `selecionar_com_sorteios` | 4 (Parte I) | roleta com sorteios fixos (didática) |
| `selecionar_pai_torneio` / `selecionar_pai_roleta` / `selecionar_pai` | 6 | seleção usada no AG completo |
| `cruzar_ox` | 5 | cruzamento que mantém a permutação válida |
| `mutacao_swap` | 6 | troca duas cidades de posição |
| `executar_algoritmo_genetico` | 6 | junta tudo e roda as gerações |
| `gerar_permutacoes` / `encontrar_otimo_global` | 7 | força bruta, só para validar o resultado |
| `repetir_execucoes` / `comparar_parametro` | 8 a 10 | experimentos com várias sementes |


## Etapa 1 — Configuração inicial

Importamos **apenas `random`**. Definimos duas funções auxiliares de impressão (`imprimir_tabela` e `barra`) para organizar a saída sem bibliotecas externas.

A semente global permite reproduzir os mesmos resultados.

#### 🔎 Entendendo o código

**O que faz:** prepara o ambiente. Importa o `random`, fixa a semente, define as constantes do problema e cria três funções auxiliares que só servem para **mostrar** resultados (não fazem parte do algoritmo genético).

**Passo a passo**
1. `random.seed(42)` faz o gerador sortear sempre a mesma sequência → qualquer pessoa que rodar o notebook obtém os mesmos números.
2. `NUM_CIDADES = 6` e `CIDADE_ORIGEM = 0` são constantes usadas em todo o código (se mudar aqui, muda em tudo).
3. `imprimir_tabela`: `zip(cabecalho, *linhas)` transforma linhas em **colunas**; o maior texto de cada coluna define a largura, e `ljust` alinha tudo.
4. `barra`: regra de três → `largura × valor / maximo` quadradinhos `█`. Valor máximo = barra cheia (40 caracteres).
5. `media`: soma dividida pela quantidade.

**O que esperar na saída:** nenhuma — a célula só define funções e constantes.


In [1]:
import random

# Semente global: permite reproduzir os mesmos resultados.
SEMENTE = 42
random.seed(SEMENTE)

NUM_CIDADES = 6
CIDADE_ORIGEM = 0

def imprimir_tabela(cabecalho, linhas):
    '''Imprime uma tabela alinhada, usando apenas print.'''
    colunas = list(zip(cabecalho, *linhas))
    larguras = [max(len(str(valor)) for valor in coluna) for coluna in colunas]

    def formatar(linha):
        return " | ".join(str(valor).ljust(larguras[i]) for i, valor in enumerate(linha))

    print(formatar(cabecalho))
    print("-+-".join("-" * largura for largura in larguras))
    for linha in linhas:
        print(formatar(linha))


def barra(valor, maximo, largura=40):
    '''Barra de texto proporcional ao valor (substitui gráficos).'''
    return "█" * round(largura * valor / maximo)


def media(valores):
    return sum(valores) / len(valores)

## Etapa 2 — Matriz de distâncias

As distâncias são **geradas aleatoriamente**. A função `gerar_matriz_distancias` sorteia as distâncias entre 10 e 100.
* A matriz é **simétrica** (a distância de
$i$ para $j$ é igual à de $j$ para $i$) e a diagonal é 0.

Para os resultados serem **reproduzíveis**, a matriz é sorteada por um gerador com semente fixa (`random.Random(SEMENTE)`).

#### 🔎 Entendendo o código

**O que faz:** cria e exibe a matriz de distâncias, onde `MATRIZ[i][j]` é a distância da cidade `i` até a cidade `j`.

**Passo a passo**
1. `[[0 for _ in range(6)] for _ in range(6)]` monta uma matriz 6×6 cheia de zeros (a diagonal já fica correta: distância de uma cidade até ela mesma é 0).
2. Os dois `for` percorrem **só o triângulo de cima** (`j` começa em `i + 1`). Cada par é sorteado **uma única vez** com `randint(10, 100)` e gravado nas duas posições (`[i][j]` e `[j][i]`). Assim a matriz sai simétrica.
3. `random.Random(SEMENTE)` cria um gerador **próprio**, com semente fixa, independente do `random` global.
4. Os `assert` conferem simetria e diagonal nula: se algo estiver errado, o código para com erro.
5. `exibir_matriz` imprime com `f"{valor:5}"` (largura 5) para as colunas ficarem alinhadas.

**Como ler a saída:** a linha é a origem e a coluna é o destino. Exemplo: linha 0, coluna 3 → distância 0→3 = **13** (e a linha 3, coluna 0 também vale 13).


In [2]:
def gerar_matriz_distancias(gerador=random):
    '''Cria uma matriz simétrica com distâncias aleatórias e diagonal zero.'''
    matriz = [[0 for _ in range(NUM_CIDADES)] for _ in range(NUM_CIDADES)]

    for i in range(NUM_CIDADES):
        for j in range(i + 1, NUM_CIDADES):
            distancia = gerador.randint(10, 100)
            matriz[i][j] = distancia
            matriz[j][i] = distancia

    return matriz


def exibir_matriz(matriz):
    '''Mostra a matriz formatada para validação visual.'''
    print("Matriz de Distâncias")
    print("    " + "".join(f"{cidade:5}" for cidade in range(NUM_CIDADES)))
    for i in range(NUM_CIDADES):
        print(f"{i:3} " + "".join(f"{matriz[i][j]:5}" for j in range(NUM_CIDADES)))


# Matriz gerada aleatoriamente no código (com semente para reprodutibilidade).
MATRIZ = gerar_matriz_distancias(random.Random(SEMENTE))

# Conferências: simetria e diagonal nula.
assert all(MATRIZ[i][j] == MATRIZ[j][i] for i in range(NUM_CIDADES) for j in range(NUM_CIDADES))
assert all(MATRIZ[i][i] == 0 for i in range(NUM_CIDADES))

exibir_matriz(MATRIZ)

Matriz de Distâncias
        0    1    2    3    4    5
  0     0   91   24   13   45   41
  1    91    0   38   27   23   96
  2    24   38    0   79   21   85
  3    13   27   79    0   64   14
  4    45   23   21   64    0   13
  5    41   96   85   14   13    0


## Etapa 3 — Representação do indivíduo e população inicial

Um indivíduo é uma **permutação** das cidades intermediárias, por exemplo `[3, 5, 1, 4, 2]`. A cidade 0 não entra no cromossomo porque é sempre o início e o fim. Assim, toda lista gerada já é uma rota válida.

Usamos lista porque facilita trocar posições na mutação. As funções recebem um `gerador` (`random.Random(semente)`), como na Aula 10, para controlar a reprodutibilidade.

#### 🔎 Entendendo o código

**O que faz:** define como um **indivíduo** (uma rota) é criado e como formar uma **população**.

**Passo a passo**
1. `gerar_individuo`: faz a lista `[0,1,2,3,4,5]`, **remove o 0** (origem fixa) e embaralha com `shuffle`. Resultado: uma permutação de 1..5, ou seja, um cromossomo válido por construção.
2. `gerar_populacao`: repete isso `tamanho_populacao` vezes e junta tudo numa lista de listas.
3. `rota_completa`: acrescenta o 0 no início e no fim, formando o trajeto real `[0, ..., 0]`.
4. O bloco final cria um exemplo para visualização.

**Detalhe de Python:** `shuffle` altera a própria lista (não devolve outra). Por isso a lista `cidades` é criada dentro da função, a cada chamada.

**Como ler a saída:** o cromossomo `[4, 2, 3, 5, 1]` vira a rota `[0, 4, 2, 3, 5, 1, 0]`.


In [3]:
def gerar_individuo(gerador=random):
    '''Cria uma rota válida: permutação aleatória das cidades 1..5.'''
    cidades = list(range(NUM_CIDADES))
    cidades.remove(CIDADE_ORIGEM)
    gerador.shuffle(cidades)
    return cidades


def gerar_populacao(tamanho_populacao, gerador=random):
    '''Cria uma lista com 'tamanho_populacao' indivíduos.'''
    populacao = []
    for _ in range(tamanho_populacao):
        populacao.append(gerar_individuo(gerador))
    return populacao


def rota_completa(individuo):
    '''Acrescenta a cidade 0 no início e no fim.'''
    return [CIDADE_ORIGEM] + individuo + [CIDADE_ORIGEM]


exemplo = gerar_individuo(random.Random(SEMENTE))
print("Cromossomo:    ", exemplo)
print("Rota completa: ", rota_completa(exemplo))

Cromossomo:     [4, 2, 3, 5, 1]
Rota completa:  [0, 4, 2, 3, 5, 1, 0]


## Etapa 4 — Função de fitness

O fitness de uma rota é a soma dos trechos:

$$
\text{custo}=d(0,c_1)+d(c_1,c_2)+d(c_2,c_3)+d(c_3,c_4)+d(c_4,c_5)+d(c_5,0).
$$

Como o objetivo é **minimizar**:
* **menor custo = melhor indivíduo**.

#### 🔎 Entendendo o código

**O que faz:** implementa o **fitness** — a distância total de uma rota. Quanto **menor**, melhor.

**Passo a passo (`calcular_fitness`)**
1. Começa em `cidade_atual = 0` com `distancia_total = 0`.
2. Para cada cidade do indivíduo: soma `matriz[cidade_atual][proxima_cidade]` e atualiza `cidade_atual`.
3. Depois do laço, soma o trecho de **volta** `cidade_atual → 0`.

`imprimir_populacao` apenas monta uma tabela (rota + distância) usando a função de fitness.

**Conferência manual:** o bloco final recalcula a distância trecho a trecho, de outro jeito, e usa `assert` para garantir que o resultado é igual ao da função.

**Como ler a saída:** `Trechos: [45, 21, 79, 14, 96, 91] -> 346` significa 0→4 = 45, 4→2 = 21, 2→3 = 79, 3→5 = 14, 5→1 = 96, 1→0 = 91. Somando: 346.


In [4]:
def calcular_fitness(individuo, matriz):
    '''Distância total da rota: 0 -> cidades do indivíduo -> 0.'''
    distancia_total = 0
    cidade_atual = CIDADE_ORIGEM

    for proxima_cidade in individuo:
        distancia_total += matriz[cidade_atual][proxima_cidade]
        cidade_atual = proxima_cidade

    distancia_total += matriz[cidade_atual][CIDADE_ORIGEM]
    return distancia_total


def imprimir_populacao(populacao, matriz):
    '''Mostra a população em tabela: rota e distância de cada indivíduo.'''
    linhas = []
    for indice, individuo in enumerate(populacao):
        linhas.append((f"c{indice + 1}", rota_completa(individuo),
                       calcular_fitness(individuo, matriz)))
    imprimir_tabela(("Indivíduo", "Rota", "Distância"), linhas)


# Conferência manual: somamos trecho a trecho, escrevendo cada parcela.
rota = rota_completa(exemplo)
trechos = [MATRIZ[rota[i]][rota[i + 1]] for i in range(len(rota) - 1)]
print("Rota:", rota)
print("Trechos:", trechos, "->", sum(trechos))
assert sum(trechos) == calcular_fitness(exemplo, MATRIZ)

Rota: [0, 4, 2, 3, 5, 1, 0]
Trechos: [45, 21, 79, 14, 96, 91] -> 346


# Parte I — Primeira Geração

Usaremos uma população pequena, de **4 indivíduos** sorteados com semente fixa.

Nesta primeira parte:

- faremos apenas uma geração;
- o cruzamento sempre ocorrerá, com o segmento fixo nas posições 2 a 3;
- a mutação será forçada em um filho, só para visualizar a troca;
- não haverá elitismo.

O objetivo não é achar a melhor rota, mas tornar cada operação visível.

#### 🔎 Entendendo o código

**O que faz:** inicia a **Parte I**: sorteia uma população pequena (4 indivíduos), calcula o custo de cada um e desenha barras para comparar.

**Passo a passo**
1. `gerar_populacao(4, random.Random(SEMENTE))` cria os indivíduos c1 a c4.
2. `imprimir_populacao` mostra rota e distância.
3. `custos_iniciais` guarda os custos numa lista (será usada nas próximas células).
4. Cada barra é proporcional ao **maior** custo, então a pior rota tem a barra mais longa.

**Como ler a saída:** o melhor indivíduo é o **c3 (309)** e o pior é o **c4 (367)**. Lembre: aqui barra menor = rota melhor.


In [5]:
populacao_inicial = gerar_populacao(4, random.Random(SEMENTE))

imprimir_populacao(populacao_inicial, MATRIZ)

print("\nDistância (barras):")
custos_iniciais = [calcular_fitness(ind, MATRIZ) for ind in populacao_inicial]
for i, custo in enumerate(custos_iniciais):
    print(f"c{i+1} {custo:4} {barra(custo, max(custos_iniciais))}")

Indivíduo | Rota                  | Distância
----------+-----------------------+----------
c1        | [0, 4, 2, 3, 5, 1, 0] | 346      
c2        | [0, 4, 3, 1, 5, 2, 0] | 341      
c3        | [0, 4, 2, 3, 1, 5, 0] | 309      
c4        | [0, 2, 3, 4, 5, 1, 0] | 367      

Distância (barras):
c1  346 ██████████████████████████████████████
c2  341 █████████████████████████████████████
c3  309 ██████████████████████████████████
c4  367 ████████████████████████████████████████


## 3. Ajuste do fitness para a roleta

A roleta dá mais chance a quem tem **aptidão maior**. Mas aqui **menor distância é melhor**. Para usar a roleta, transformamos o custo em um peso que cresce quando o custo diminui:

$$
F_i=\frac{1}{\text{custo}_i}, \qquad p_i=\frac{F_i}{\sum_j F_j}.
$$

Esse ajuste:

- preserva a ordem (menor custo, maior peso);
- mantém todos os pesos positivos;
- **não** altera o custo original, é usado só para calcular as probabilidades.

#### 🔎 Entendendo o código

**O que faz:** transforma o custo em **probabilidade de ser escolhido** na roleta.

**Por que precisa disso?** A roleta dá mais chance a quem tem valor **maior**, mas no nosso problema o melhor é o **menor** custo. A solução é inverter: `peso = 1 / custo`.

**Passo a passo**
1. `calcular_pesos_roleta`: devolve `1/custo` para cada indivíduo (custo menor → peso maior).
2. `calcular_probabilidades`: divide cada peso pela soma dos pesos (normalização) → as probabilidades somam 1.
3. A tabela mostra custo, peso, probabilidade e uma barra proporcional.

**Como ler a saída:** c3 (custo 309) tem a maior fatia, **27,46%**; c4 (custo 367) tem a menor, **23,12%**. As fatias ficam parecidas porque os custos são parecidos. A última linha confirma que a soma é 1.


In [6]:
def calcular_pesos_roleta(custos):
    '''Converte custos (minimizar) em pesos positivos: quanto menor o custo, maior o peso.'''
    return [1 / custo for custo in custos]


def calcular_probabilidades(pesos):
    '''Normaliza os pesos: as probabilidades resultantes somam 1.'''
    soma = sum(pesos)
    return [peso / soma for peso in pesos]


pesos_iniciais = calcular_pesos_roleta(custos_iniciais)
probabilidades_iniciais = calcular_probabilidades(pesos_iniciais)

linhas = []
for i, ind in enumerate(populacao_inicial):
    linhas.append((f"c{i+1}", custos_iniciais[i], f"{pesos_iniciais[i]:.6f}",
                   f"{100 * probabilidades_iniciais[i]:.2f}%",
                   barra(probabilidades_iniciais[i], 1, 40)))
imprimir_tabela(("Indivíduo", "Distância", "Peso (1/custo)", "Probabilidade", "Roleta"), linhas)
print("\nSoma das probabilidades:", round(sum(probabilidades_iniciais), 6))

Indivíduo | Distância | Peso (1/custo) | Probabilidade | Roleta     
----------+-----------+----------------+---------------+------------
c1        | 346       | 0.002890       | 24.53%        | ██████████ 
c2        | 341       | 0.002933       | 24.89%        | ██████████ 
c3        | 309       | 0.003236       | 27.46%        | ███████████
c4        | 367       | 0.002725       | 23.12%        | █████████  

Soma das probabilidades: 1.0


## 4. Seleção por roleta

As faixas cumulativas são

$$
[0,p_1),\quad[p_1,p_1+p_2),\quad[p_1+p_2,\,p_1+p_2+p_3),\quad[p_1+p_2+p_3,\,1].
$$

Para tornar o exemplo reproduzível, usaremos sorteios fixos:

$$
r_1=0{,}30,\quad r_2=0{,}70,\quad r_3=0{,}90,\quad r_4=0{,}50.
$$

A seleção ocorre **com reposição**: o mesmo indivíduo pode aparecer mais de uma vez entre os pais.

#### 🔎 Entendendo o código

**O que faz:** simula a **roleta** com sorteios fixos (`r`), para o resultado ser conferido à mão.

**Passo a passo (`selecionar_com_sorteios`)**
1. Monta os **limites acumulados** somando as probabilidades uma a uma: cada indivíduo ocupa uma faixa entre 0 e 1.
2. Para cada sorteio `r`, o `while` avança de faixa em faixa **enquanto `r` for maior que o limite atual**. Quando para, aquele é o escolhido.
3. `populacao[indice].copy()` guarda uma **cópia** do pai. Assim, mudar o filho depois não altera a população original.
4. Os índices escolhidos formam o **mating pool** (grupo de pais).

**Como ler a saída:** os limites são `[0.2453, 0.4941, 0.7688, 1.0]`.
- r = 0,30 está entre 0,2453 e 0,4941 → **c2**
- r = 0,70 → entre 0,4941 e 0,7688 → **c3**
- r = 0,90 → acima de 0,7688 → **c4**
- r = 0,50 → logo acima de 0,4941 → **c3**

A seleção é **com reposição**: c3 foi escolhido duas vezes e c1 não foi escolhido nenhuma.


In [7]:
def selecionar_com_sorteios(populacao, probabilidades, sorteios):
    '''Seleciona indivíduos usando valores aleatórios já conhecidos.'''
    limites_acumulados = []
    soma = 0
    for p in probabilidades:
        soma += p
        limites_acumulados.append(soma)

    indices_escolhidos = []
    for sorteio in sorteios:
        indice = 0
        # Avança enquanto o sorteio estiver fora da faixa atual.
        while sorteio > limites_acumulados[indice]:
            indice += 1
        indices_escolhidos.append(indice)

    pais = [populacao[indice].copy() for indice in indices_escolhidos]
    return pais, indices_escolhidos, limites_acumulados


sorteios_exemplo = [0.30, 0.70, 0.90, 0.50]
pais, indices_pais, acumuladas = selecionar_com_sorteios(
    populacao_inicial, probabilidades_iniciais, sorteios_exemplo
)

print("Limites acumulados:", [round(v, 4) for v in acumuladas])
print("Mating pool:", [f"c{i + 1}" for i in indices_pais], "\n")

imprimir_tabela(
    ("Sorteio r", "Selecionado", "Cromossomo"),
    [(r, f"c{i + 1}", pai) for r, i, pai in zip(sorteios_exemplo, indices_pais, pais)],
)

Limites acumulados: [0.2453, 0.4941, 0.7688, 1.0]
Mating pool: ['c2', 'c3', 'c4', 'c3'] 

Sorteio r | Selecionado | Cromossomo     
----------+-------------+----------------
0.3       | c2          | [4, 3, 1, 5, 2]
0.7       | c3          | [4, 2, 3, 1, 5]
0.9       | c4          | [2, 3, 4, 5, 1]
0.5       | c3          | [4, 2, 3, 1, 5]


## 5. Cruzamento OX (*Order Crossover*)

Em um problema de permutação, o cruzamento convencional de um ponto pode gerar cidades repetidas e deixar outras ausentes. O cruzamento OX evita esse problema, mantendo cada cidade exatamente uma vez no cromossomo.

O funcionamento ocorre em três passos:

* Sorteia dois pontos, inicio e fim, e copia o segmento entre essas posições de um dos pais para o filho.

* Percorre o outro pai a partir do final do segmento, mantendo a ordem das cidades e ignorando aquelas que já estão presentes no filho.

* Preenche as posições vazias a partir de fim + 1, voltando ao início do cromossomo quando necessário, até completar o filho.

### Exemplo à mão

Pai 1 = `[1, 2, 3, 4, 5]`, Pai 2 = `[5, 3, 1, 2, 4]`, segmento nas posições 1 a 3.

- cópia do pai 1: filho = `[_, 2, 3, 4, _]`
- percorrendo o pai 2 a partir da posição 4: `4, 5, 3, 1, 2`; removendo `2, 3, 4`, sobram `5, 1`
- preenchendo as posições `4` e `0`: filho = `[1, 2, 3, 4, 5]`

O código abaixo permite fixar o segmento (para reproduzir exemplos) ou sorteá-lo.

#### 🔎 Entendendo o código

**O que faz:** implementa o cruzamento **OX** (*Order Crossover*) e confere o exemplo feito à mão.

**Passo a passo (`cruzar_ox`)**
1. Se `inicio` e `fim` não forem informados, sorteia dois cortes com `sample` e ordena com `sorted` (o menor vira `inicio`).
2. Dentro de `criar_filho(primeiro, segundo)`:
   - `filho = [None] * tamanho` cria o filho vazio;
   - `filho[inicio:fim+1] = primeiro[inicio:fim+1]` copia o **segmento** do primeiro pai;
   - `cidades_presentes` (um `set`) guarda o que já foi copiado;
   - `restante` percorre o segundo pai **começando logo após o segmento** (`segundo[fim+1:] + segundo[:fim+1]`) e ignora as cidades já presentes;
   - `posicoes` são as vagas livres, também a partir de `fim+1` e voltando ao começo (`range(fim+1, tamanho)` depois `range(0, inicio)`);
   - `zip(posicoes, restante)` preenche as vagas na ordem.
3. A função devolve **dois filhos**: `criar_filho(pai1, pai2)` e `criar_filho(pai2, pai1)` (os pais trocam de papel).

**Exemplo (pai 1 = `[1,2,3,4,5]`, pai 2 = `[5,3,1,2,4]`, segmento 1 a 3):**

| Passo | Filho 1 |
|---|---|
| copia segmento do pai 1 | `[_, 2, 3, 4, _]` |
| lê o pai 2 a partir da posição 4 | `4, 5, 3, 1, 2` |
| tira as já presentes (2, 3, 4) | sobram `5, 1` |
| preenche posições 4 e 0 | `[1, 2, 3, 4, 5]` |

**Como ler a saída:** `Filho 1: [1, 2, 3, 4, 5]` (igual ao feito à mão, o `assert` confirma) e `Filho 2: [4, 3, 1, 2, 5]`. Nenhum dos dois repete ou perde cidade.


In [8]:
def cruzar_ox(pai1, pai2, gerador=random, inicio=None, fim=None):
    '''Order Crossover: retorna dois filhos que continuam sendo permutações.'''
    tamanho = len(pai1)

    # Se o segmento não for informado, sorteia dois cortes.
    if inicio is None or fim is None:
        inicio, fim = sorted(gerador.sample(range(tamanho), 2))

    def criar_filho(primeiro, segundo):
        filho = [None] * tamanho

        # Copia um segmento do primeiro pai.
        filho[inicio:fim + 1] = primeiro[inicio:fim + 1]
        cidades_presentes = set(filho[inicio:fim + 1])

        # Percorre o segundo pai a partir do fim do segmento,
        # mantendo a ordem e ignorando cidades já copiadas.
        restante = [
            cidade for cidade in segundo[fim + 1:] + segundo[:fim + 1]
            if cidade not in cidades_presentes
        ]

        posicoes = list(range(fim + 1, tamanho)) + list(range(0, inicio))
        for posicao, cidade in zip(posicoes, restante):
            filho[posicao] = cidade

        return filho

    return criar_filho(pai1, pai2), criar_filho(pai2, pai1)


# Confere o exemplo feito à mão.
f1, f2 = cruzar_ox([1, 2, 3, 4, 5], [5, 3, 1, 2, 4], inicio=1, fim=3)
print("Filho 1:", f1, "| Filho 2:", f2)
assert f1 == [1, 2, 3, 4, 5]

Filho 1: [1, 2, 3, 4, 5] | Filho 2: [4, 3, 1, 2, 5]


## 6. Mutação swap

Com probabilidade `taxa_mutacao`, **troca duas cidades de posição**. Isso mantém a rota válida e introduz variações que o cruzamento sozinho não cria (o cruzamento só reorganiza o que já existe).

Na geração didática, vamos **forçar** a mutação (`taxa_mutacao=1.0`) no primeiro filho, apenas para ver o efeito.

#### 🔎 Entendendo o código

**O que faz:** define a **mutação swap** e aplica uma geração completa da Parte I (cruzamento + mutação), mostrando a nova população.

**Passo a passo (`mutacao_swap`)**
1. `individuo.copy()` evita alterar o original.
2. `gerador.random() < taxa_mutacao` sorteia um número entre 0 e 1: se for menor que a taxa, a mutação acontece (taxa 0,10 → 10% de chance; taxa 1.0 → sempre).
3. `sample(range(len(mutante)), 2)` escolhe **duas posições diferentes**, e `mutante[i], mutante[j] = mutante[j], mutante[i]` troca as cidades.

**Passo a passo (bloco de teste)**
1. Forma os pares de pais do mating pool: (1º, 2º) e (3º, 4º).
2. Cruza cada par com segmento fixo nas posições 2 a 3 → 4 filhos.
3. Força a mutação (`taxa_mutacao=1.0`) **só no filho 1**, para vermos o efeito.
4. O `assert sorted(filho) == [1, 2, 3, 4, 5]` garante que todos continuam sendo rotas válidas.
5. Imprime a nova população com as distâncias.

**Como ler a saída:** o filho 1 era `[2, 3, 1, 5, 4]` e virou `[1, 3, 2, 5, 4]` (trocaram as posições 0 e 2: as cidades 2 e 1). Repare que **c2 e c4 ficaram iguais** (distância 161): com população pequena, a diversidade cai rápido.


In [9]:
def mutacao_swap(individuo, taxa_mutacao, gerador=random):
    '''Com a probabilidade dada, troca duas posições do indivíduo.'''
    mutante = individuo.copy()

    if gerador.random() < taxa_mutacao:
        i, j = gerador.sample(range(len(mutante)), 2)
        mutante[i], mutante[j] = mutante[j], mutante[i]

    return mutante


gerador_exemplo = random.Random(SEMENTE)

# Pares de pais do mating pool: (1º, 2º) e (3º, 4º), segmento nas posições 2 a 3.
filho_1, filho_2 = cruzar_ox(pais[0], pais[1], inicio=2, fim=3)
filho_3, filho_4 = cruzar_ox(pais[2], pais[3], inicio=2, fim=3)

print("Pais 1 e 2:", pais[0], pais[1])
print("  Filho 1:", filho_1, "| Filho 2:", filho_2)
print("Pais 3 e 4:", pais[2], pais[3])
print("  Filho 3:", filho_3, "| Filho 4:", filho_4)

# Mutação forçada apenas no filho 1.
filho_1_mutado = mutacao_swap(filho_1, taxa_mutacao=1.0, gerador=gerador_exemplo)
print("\nFilho 1 antes da mutação: ", filho_1)
print("Filho 1 depois da mutação:", filho_1_mutado)

nova_populacao = [filho_1_mutado, filho_2, filho_3, filho_4]

for filho in nova_populacao:
    assert sorted(filho) == [1, 2, 3, 4, 5], "rota inválida!"

print("\nNova população:")
imprimir_populacao(nova_populacao, MATRIZ)

Pais 1 e 2: [4, 3, 1, 5, 2] [4, 2, 3, 1, 5]
  Filho 1: [2, 3, 1, 5, 4] | Filho 2: [4, 5, 3, 1, 2]
Pais 3 e 4: [2, 3, 4, 5, 1] [4, 2, 3, 1, 5]
  Filho 3: [3, 1, 4, 5, 2] | Filho 4: [4, 5, 3, 1, 2]

Filho 1 antes da mutação:  [2, 3, 1, 5, 4]
Filho 1 depois da mutação: [1, 3, 2, 5, 4]

Nova população:
Indivíduo | Rota                  | Distância
----------+-----------------------+----------
c1        | [0, 1, 3, 2, 5, 4, 0] | 340      
c2        | [0, 4, 5, 3, 1, 2, 0] | 161      
c3        | [0, 3, 1, 4, 5, 2, 0] | 185      
c4        | [0, 4, 5, 3, 1, 2, 0] | 161      


#### 🔎 Entendendo o código

**O que faz:** compara a população **antes** e **depois** da primeira geração.

**Passo a passo**
1. Calcula os custos das duas populações.
2. `maximo` (o maior custo entre as duas) serve de referência para as barras, para que fiquem comparáveis.
3. Imprime as barras de cada indivíduo (antes × depois) e depois o melhor custo e o custo médio de cada geração.

**Como ler a saída:** o melhor custo caiu de **309 para 161** e a média de **340,8 para 211,8**. Houve melhora, mas isso **não é garantido**: não usamos elitismo, então a melhor rota poderia ter sido perdida.


In [10]:
# Comparação entre as duas gerações.
custos_antes = [calcular_fitness(ind, MATRIZ) for ind in populacao_inicial]
custos_depois = [calcular_fitness(ind, MATRIZ) for ind in nova_populacao]
maximo = max(custos_antes + custos_depois)

print("Distância por indivíduo (menor é melhor):")
for i in range(4):
    print(f"Indivíduo {i+1}  antes  {custos_antes[i]:4} {barra(custos_antes[i], maximo)}")
    print(f"             depois {custos_depois[i]:4} {barra(custos_depois[i], maximo)}")

print(f"\nMelhor distância antes:  {min(custos_antes)}")
print(f"Melhor distância depois: {min(custos_depois)}")
print(f"Distância média antes:   {media(custos_antes):.1f}")
print(f"Distância média depois:  {media(custos_depois):.1f}")

Distância por indivíduo (menor é melhor):
Indivíduo 1  antes   346 ██████████████████████████████████████
             depois  340 █████████████████████████████████████
Indivíduo 2  antes   341 █████████████████████████████████████
             depois  161 ██████████████████
Indivíduo 3  antes   309 ██████████████████████████████████
             depois  185 ████████████████████
Indivíduo 4  antes   367 ████████████████████████████████████████
             depois  161 ██████████████████

Melhor distância antes:  309
Melhor distância depois: 161
Distância média antes:   340.8
Distância média depois:  211.8


### O que aprendemos com uma geração?

- A roleta favorece rotas curtas, mas ainda pode escolher rotas ruins e deixar de escolher as boas.
- O OX combina trechos dos pais **sem repetir nem perder cidades**.
- O cruzamento só reaproveita material que já existe; a mutação cria variações novas.
- Sem elitismo, **não há garantia** de que a melhor rota sobreviva: compare a "melhor distância" antes e depois.
- Com apenas 4 indivíduos há pouca diversidade; o exemplo serve para explicar os cálculos.

# Parte II — Algoritmo Genético completo (Etapa 6)

Agora usamos uma população maior e acrescentamos:

- **seleção** por torneio ou roleta (escolhida por parâmetro);
- cruzamento probabilístico;
- mutação;
- **elitismo**;
- várias gerações;
- acompanhamento do melhor custo, custo médio e diversidade.

## Escolhas didáticas

| Parâmetro | Valor inicial | Interpretação |
|---|---:|---|
| Tamanho da população | 60 | Existem sessenta rotas por geração |
| Gerações | 100 | O algoritmo evolui por 100 ciclos (limite de ciclos evolutivos) |
| Taxa de cruzamento | 0,80 | Há 80% de chance de cruzar cada par de pais. |
| Taxa de mutação | 0,10 | Cada filho tem 10% de chance de sofrer mutação. |
| Elitismo | 1 |A melhor rota é preservada na próxima geração.|
| Seleção | torneio (3 participantes) | escolhe o menor custo entre 3 sorteados |



### Etapa 6A — Seleção por torneio

Sorteia participantes e escolhe aquele cuja rota tem o menor custo.

#### 🔎 Entendendo o código

**O que faz:** seleção por **torneio** — sorteia alguns indivíduos e o de **menor custo** vence.

**Passo a passo**
1. `gerador.sample(range(len(populacao)), k)` sorteia `k` **índices diferentes** (`k` = `tamanho_torneio`, padrão 3). O `min(..., len(populacao))` evita erro se a população for menor que `k`.
2. `min(indices, key=lambda i: custos[i])` devolve o índice cujo custo é o menor entre os sorteados. (`lambda` é uma função curta que diz "compare pelo custo".)
3. Devolve uma **cópia** do vencedor.

**Ideia-chave:** quanto maior o torneio, mais forte é a pressão para escolher rotas boas. Mesmo assim, uma rota ruim ainda pode ganhar se só sortear rotas ruins.

**O que esperar na saída:** nenhuma (definição de função).


In [11]:
def selecionar_pai_torneio(populacao, custos, gerador, tamanho_torneio=3):
    """Sorteia participantes e devolve uma cópia do que tem MENOR custo."""
    indices = gerador.sample(
        range(len(populacao)),
        min(tamanho_torneio, len(populacao))
    )
    vencedor = min(indices, key=lambda i: custos[i])
    return populacao[vencedor].copy()


### Etapa 6B — Seleção por roleta

Atribui pesos às rotas; rotas de menor custo devem ter maior chance de seleção.

#### 🔎 Entendendo o código

**O que faz:** seleção por **roleta**, agora de forma automática (sem sorteios fixos).

**Passo a passo**
1. `calcular_pesos_roleta(custos)` calcula `1/custo` de cada indivíduo (a mesma conta da seção 3).
2. `gerador.choices(range(n), weights=pesos, k=1)` sorteia **um** índice com probabilidade proporcional aos pesos. É a mesma roleta das faixas acumuladas, só que o Python faz essa conta por dentro.
3. `choices` devolve uma **lista**, por isso o `[0]` no final pega o único elemento.
4. Devolve uma cópia do indivíduo escolhido.

**O que esperar na saída:** nenhuma (definição de função).


In [12]:
def selecionar_pai_roleta(populacao, custos, gerador):
    """Seleciona uma cópia de um indivíduo por roleta (peso = 1/custo)."""
    pesos = calcular_pesos_roleta(custos)
    indice = gerador.choices(
        range(len(populacao)),
        weights=pesos,
        k=1
    )[0]
    return populacao[indice].copy()


### Etapa 6C — Escolha do método de seleção

Encaminha a seleção para torneio ou roleta, conforme o parâmetro configurado.

#### 🔎 Entendendo o código

**O que faz:** funciona como uma "chave seletora": recebe o nome do método e chama a função certa.

**Como funciona:** se `metodo == "roleta"` usa a roleta; **em qualquer outro caso**, usa o torneio. Assim o algoritmo principal não precisa saber os detalhes de cada seleção; basta trocar o parâmetro `metodo_selecao`.

**O que esperar na saída:** nenhuma (definição de função).


In [13]:
def selecionar_pai(populacao, custos, gerador, metodo, tamanho_torneio):
    """Escolhe o método de seleção conforme o parâmetro."""
    if metodo == "roleta":
        return selecionar_pai_roleta(populacao, custos, gerador)
    return selecionar_pai_torneio(
        populacao, custos, gerador, tamanho_torneio
    )


#### 🔎 Entendendo o código

**O que faz:** é o **algoritmo genético completo**. Junta tudo o que foi feito até aqui e repete o ciclo por várias gerações.

**Visão geral do ciclo**
```
população inicial
      ↓
┌─► avalia (custos) e registra no histórico
│        ↓
│   elitismo: copia os melhores para a nova população
│        ↓
│   enquanto a nova população não estiver cheia:
│        escolhe 2 pais (seleção) → cruza (ou copia) → muta cada filho
│        ↓
└── nova população substitui a anterior
```

**Passo a passo**
1. `gerador = random.Random(semente)` → todas as decisões aleatórias saem de um único gerador, o que torna a execução reproduzível. Em seguida cria a população inicial.
2. O laço `for geracao in range(numero_geracoes + 1)` roda uma vez a mais para também **avaliar a última população**.
3. **Avaliação:** calcula `custos`, acha o melhor indivíduo e mede a **diversidade** (`{tuple(ind) for ind in populacao}`: o `set` remove repetidos; usamos `tuple` porque lista não pode entrar em `set`). Tudo vai para o `historico` (um dicionário por geração).
4. Se for a última geração, o `break` encerra (não precisa gerar descendentes).
5. **Elitismo:** `sorted(..., key=custo)` ordena os índices do melhor para o pior e copia os `elitismo` primeiros para a nova população.
6. **Reprodução:** enquanto faltar gente, escolhe `pai1` e `pai2`. Com probabilidade `taxa_cruzamento` aplica o OX; senão, os filhos são cópias dos pais. Depois aplica a mutação swap em cada filho.
7. `filho2` só entra se ainda houver vaga (necessário quando a população tem tamanho ímpar ou sobra espaço depois do elitismo).
8. A nova população vira a população atual e o ciclo recomeça. No final, retorna `(populacao, historico)`.

**O que esperar na saída:** nenhuma (definição de função). Ela será chamada na Etapa 8.


In [14]:
def executar_algoritmo_genetico(
    matriz,
    tamanho_populacao=60,
    numero_geracoes=100,
    taxa_cruzamento=0.80,
    taxa_mutacao=0.10,
    elitismo=1,
    metodo_selecao="torneio",
    tamanho_torneio=3,
    semente=42,
):
    '''Executa o AG e retorna a população final e o histórico (lista de dicionários).'''

    gerador = random.Random(semente)
    populacao = gerar_populacao(tamanho_populacao, gerador)
    historico = []

    for geracao in range(numero_geracoes + 1):
        custos = [calcular_fitness(ind, matriz) for ind in populacao]
        melhor_indice = min(range(len(populacao)), key=lambda i: custos[i])

        # Diversidade simples: quantidade de rotas diferentes.
        diversidade = len({tuple(ind) for ind in populacao})

        historico.append({
            "geracao": geracao,
            "melhor_custo": min(custos),
            "custo_medio": media(custos),
            "pior_custo": max(custos),
            "diversidade": diversidade,
            "melhor_individuo": populacao[melhor_indice].copy(),
        })

        # A última população é avaliada, mas não precisa gerar descendentes.
        if geracao == numero_geracoes:
            break

        nova_populacao = []

        # Elitismo: preserva os melhores antes de produzir os filhos.
        ordem = sorted(range(len(populacao)), key=lambda i: custos[i])
        for indice in ordem[:elitismo]:
            nova_populacao.append(populacao[indice].copy())

        while len(nova_populacao) < tamanho_populacao:
            pai1 = selecionar_pai(populacao, custos, gerador, metodo_selecao, tamanho_torneio)
            pai2 = selecionar_pai(populacao, custos, gerador, metodo_selecao, tamanho_torneio)

            if gerador.random() < taxa_cruzamento:
                filho1, filho2 = cruzar_ox(pai1, pai2, gerador)
            else:
                # Sem cruzamento, os pais são copiados.
                filho1, filho2 = pai1.copy(), pai2.copy()

            filho1 = mutacao_swap(filho1, taxa_mutacao, gerador)
            filho2 = mutacao_swap(filho2, taxa_mutacao, gerador)

            nova_populacao.append(filho1)
            if len(nova_populacao) < tamanho_populacao:
                nova_populacao.append(filho2)

        populacao = nova_populacao

    return populacao, historico

### Etapa 7 — Configuração dos parâmetros

Altere os valores abaixo para experimentar diferentes configurações do algoritmo.

#### 🔎 Entendendo o código

**O que faz:** reúne todos os parâmetros do experimento em um só lugar.

**Como usar:** altere os valores e rode as células seguintes de novo para ver o efeito. Sugestões de teste: `TAXA_MUTACAO = 0.0`, `ELITISMO = 0`, `METODO_SELECAO = "roleta"`, `SEMENTE = 1`.

**O que esperar na saída:** nenhuma (só atribui valores às variáveis).


In [15]:
# Parâmetros concentrados em um único lugar para facilitar os experimentos.
TAMANHO_POPULACAO = 60
NUMERO_GERACOES = 100
TAXA_CRUZAMENTO = 0.80
TAXA_MUTACAO = 0.10
ELITISMO = 1
METODO_SELECAO = "torneio"   # "torneio" ou "roleta"
SEMENTE = 42


### Etapa 8 — Execução do algoritmo

Executa o algoritmo genético com os parâmetros definidos acima.

#### 🔎 Entendendo o código

**O que faz:** executa o algoritmo com os parâmetros da célula anterior.

**O que ela devolve:**
- `populacao_final`: as rotas da última geração;
- `historico`: uma lista com 101 dicionários (gerações 0 a 100), cada um com melhor/médio/pior custo, diversidade e melhor indivíduo.

**O que esperar na saída:** nenhuma impressão. Os dados ficam guardados nas variáveis e são mostrados nas próximas etapas.


In [16]:
populacao_final, historico = executar_algoritmo_genetico(
    MATRIZ,
    tamanho_populacao=TAMANHO_POPULACAO,
    numero_geracoes=NUMERO_GERACOES,
    taxa_cruzamento=TAXA_CRUZAMENTO,
    taxa_mutacao=TAXA_MUTACAO,
    elitismo=ELITISMO,
    metodo_selecao=METODO_SELECAO,
    semente=SEMENTE,
)


### Etapa 9 — Resultado final

Mostra a melhor rota registrada na última geração, a distância total e a diversidade da população final.

#### 🔎 Entendendo o código

**O que faz:** lê o **último** registro do histórico (`historico[-1]`) e mostra o resultado final.

**Como ler a saída:** o AG encontrou a rota `[0, 2, 4, 1, 3, 5, 0]` com distância **150**, e a população final tem **5 rotas diferentes** (de 60 indivíduos). Ainda **não sabemos** se 150 é o melhor possível: isso é verificado na seção 7.


In [17]:
ultima = historico[-1]

print("===== RESULTADO FINAL =====")
print("Melhor rota encontrada:", rota_completa(ultima["melhor_individuo"]))
print("Distância total:", ultima["melhor_custo"])
print("Diversidade final:", ultima["diversidade"], "rotas diferentes")


===== RESULTADO FINAL =====
Melhor rota encontrada: [0, 2, 4, 1, 3, 5, 0]
Distância total: 150
Diversidade final: 5 rotas diferentes


### Etapa 10 — Histórico das gerações

A tabela seguinte acompanha o melhor custo, o custo médio, o pior custo e a diversidade ao longo das gerações.

#### 🔎 Entendendo o código

**O que faz:** mostra a evolução em uma tabela, **uma linha a cada 10 gerações** (`h["geracao"] % 10 == 0`), com barras proporcionais ao pior custo da geração 0.

**Como ler a saída**
- **Melhor:** ficou em 150 durante todo o experimento; essa rota já existia na geração 0.
- **Médio:** caiu de **282,4** para cerca de 153 na geração 10 → a população inteira ficou parecida com as boas rotas.
- **Diversidade:** caiu de **48** para **4** já na geração 10. Isso é a população "convergindo".
- **Pior:** continua oscilando (264 a 341), por causa da mutação que às vezes cria rotas ruins.

**Pergunta para a próxima seção:** será que 150 é mesmo a melhor rota possível?


In [18]:
# Evolução ao longo das gerações (a cada 10), com barras de texto.
pior_inicial = historico[0]["pior_custo"]

imprimir_tabela(
    ("Geração", "Melhor", "Médio", "Pior", "Diversidade", "Melhor (barra)"),
    [
        (h["geracao"], h["melhor_custo"], f"{h['custo_medio']:.1f}", h["pior_custo"],
         h["diversidade"], barra(h["melhor_custo"], pior_inicial))
        for h in historico if h["geracao"] % 10 == 0
    ],
)

Geração | Melhor | Médio | Pior | Diversidade | Melhor (barra)
--------+--------+-------+------+-------------+---------------
0       | 150    | 282.4 | 460  | 48          | █████████████ 
10      | 150    | 153.2 | 264  | 4           | █████████████ 
20      | 150    | 156.1 | 269  | 7           | █████████████ 
30      | 150    | 158.7 | 341  | 6           | █████████████ 
40      | 150    | 157.6 | 269  | 7           | █████████████ 
50      | 150    | 163.1 | 273  | 8           | █████████████ 
60      | 150    | 157.2 | 207  | 5           | █████████████ 
70      | 150    | 160.1 | 273  | 8           | █████████████ 
80      | 150    | 163.9 | 341  | 8           | █████████████ 
90      | 150    | 155.2 | 264  | 5           | █████████████ 
100     | 150    | 161.9 | 341  | 5           | █████████████ 


## 7. Verificação pelo espaço de busca completo

Como existem só $5!=120$ rotas, podemos enumerar todas e descobrir o **ótimo global**. Como não podemos usar `itertools`, geramos as permutações com uma **função recursiva própria**. Essa enumeração serve apenas para **validar** o experimento; ela não faz parte do AG.

Com $n$ cidades o espaço cresce como $(n-1)!$: para 15 cidades já seriam cerca de $8{,}7\times10^{10}$ rotas, e a enumeração deixa de ser viável. É aí que heurísticas como o AG são úteis.

#### 🔎 Entendendo o código

**O que faz:** testa **todas as rotas possíveis** para descobrir o ótimo global e comparar com o AG. Isso é só validação, não faz parte do algoritmo genético.

**Passo a passo**
1. `gerar_permutacoes` é **recursiva**:
   - caso base: lista com 0 ou 1 elemento → só existe uma permutação;
   - caso geral: para cada posição `i`, fixa `lista[i]` na frente e gera as permutações do **resto** (`lista[:i] + lista[i+1:]`), juntando as duas partes.
2. `encontrar_otimo_global` calcula o custo de cada uma das 120 permutações e guarda a de menor custo (`float("inf")` é um "infinito" para começar a comparação).
3. Compara o melhor do AG com o ótimo e imprime a diferença.

**Como ler a saída:** o ótimo global é `[0, 2, 1, 4, 5, 3, 0]` com custo **125**. O AG (com semente 42) parou em **150**, ou seja, **25 acima**. Esse é um exemplo de **convergência prematura**: a população ficou parecida demais antes de achar a melhor rota.

**Observação:** como a matriz é simétrica, uma rota e a sua inversa têm o mesmo custo; por isso a rota do AG poderia vir invertida e ainda assim ser ótima.


In [19]:
def gerar_permutacoes(lista):
    '''Gera todas as permutações de uma lista, recursivamente.'''
    if len(lista) <= 1:
        return [lista.copy()]

    resultado = []
    for i in range(len(lista)):
        resto = lista[:i] + lista[i + 1:]
        for permutacao in gerar_permutacoes(resto):
            resultado.append([lista[i]] + permutacao)
    return resultado


def encontrar_otimo_global(matriz):
    '''Enumera todas as permutações das cidades 1..5.'''
    melhor_individuo = None
    melhor_custo = float("inf")
    todos_os_custos = []

    cidades = [c for c in range(NUM_CIDADES) if c != CIDADE_ORIGEM]
    for individuo in gerar_permutacoes(cidades):
        custo = calcular_fitness(individuo, matriz)
        todos_os_custos.append(custo)

        if custo < melhor_custo:
            melhor_custo = custo
            melhor_individuo = individuo

    return melhor_individuo, melhor_custo, todos_os_custos


otimo_global, custo_otimo, todos_os_custos = encontrar_otimo_global(MATRIZ)
melhor_ag = ultima["melhor_individuo"]
custo_ag = calcular_fitness(melhor_ag, MATRIZ)

assert len(todos_os_custos) == 120

print("Rotas possíveis:", len(todos_os_custos))
print("Ótimo global por enumeração:", rota_completa(otimo_global))
print("Custo ótimo global:", custo_otimo)
print("Melhor rota do AG:", rota_completa(melhor_ag))
print("Custo do AG:", custo_ag)

# Obs.: como a matriz é simétrica, uma rota e a sua inversa têm o mesmo custo;
# por isso a rota do AG pode ser a inversa da encontrada por enumeração.
if custo_ag == custo_otimo:
    print("Resultado: nesta execução, o AG encontrou o ótimo global.")
else:
    print(f"Resultado: o AG ficou {custo_ag - custo_otimo} acima do ótimo global.")

Rotas possíveis: 120
Ótimo global por enumeração: [0, 2, 1, 4, 5, 3, 0]
Custo ótimo global: 125
Melhor rota do AG: [0, 2, 4, 1, 3, 5, 0]
Custo do AG: 150
Resultado: o AG ficou 25 acima do ótimo global.


#### 🔎 Entendendo o código

**O que faz:** monta um **histograma de texto** mostrando em que faixa de custo caem as 120 rotas possíveis, e marca onde estão o ótimo e o AG.

**Passo a passo**
1. Pega o menor e o maior custo e define faixas de largura 40.
2. Para cada custo, calcula a faixa dele: `menor + ((custo - menor) // 40) * 40`.
3. Conta quantas rotas caem em cada faixa usando um dicionário (`faixas.get(faixa, 0) + 1`).
4. Imprime uma barra por faixa e coloca `<- ótimo global` e `<- AG` nas faixas correspondentes.

**Como ler a saída:** o ótimo (125) e o AG (150) estão na **mesma faixa, 125–164**, que tem só 8 das 120 rotas. Ou seja, o AG achou uma rota entre as melhores, mas não a melhor. (Como as faixas têm largura 40, o gráfico não separa 125 de 150.)


In [20]:
# Onde a solução do AG está entre todas as 120 rotas possíveis? (histograma de texto)
menor, maior = min(todos_os_custos), max(todos_os_custos)
largura_faixa = 40
faixas = {}
for custo in todos_os_custos:
    inicio_faixa = menor + ((custo - menor) // largura_faixa) * largura_faixa
    faixas[inicio_faixa] = faixas.get(inicio_faixa, 0) + 1

print("Distribuição do custo das 120 rotas possíveis")
for inicio_faixa in sorted(faixas):
    fim_faixa = inicio_faixa + largura_faixa - 1
    marcador = ""
    if inicio_faixa <= custo_otimo <= fim_faixa:
        marcador += "  <- ótimo global"
    if inicio_faixa <= custo_ag <= fim_faixa:
        marcador += "  <- AG"
    print(f"{inicio_faixa:4}-{fim_faixa:<4} {faixas[inicio_faixa]:3} {barra(faixas[inicio_faixa], max(faixas.values()))}{marcador}")

Distribuição do custo das 120 rotas possíveis
 125-164    8 ███████████  <- ótimo global  <- AG
 165-204   14 ███████████████████
 205-244   20 ███████████████████████████
 245-284   30 ████████████████████████████████████████
 285-324   22 █████████████████████████████
 325-364   18 ████████████████████████
 365-404    6 ████████
 445-484    2 ███


## 8. Uma execução é suficiente?

Não. O AG tem decisões aleatórias. Para avaliar o comportamento, repetimos o experimento com 30 sementes e observamos quantas execuções encontram o ótimo global.

Com população 60 e 100 gerações, o AG tem muita folga para um espaço de apenas 120 rotas. Por isso, a partir daqui os experimentos usam uma configuração **mais restrita** (população 8 e 8 gerações), em que as diferenças entre parâmetros ficam visíveis.

#### 🔎 Entendendo o código

**O que faz:** repete o AG com **30 sementes diferentes** e mede em quantas execuções ele encontra o ótimo (**taxa de sucesso**), em duas configurações.

**Passo a passo**
1. `repetir_execucoes(sementes, **parametros)`: o `**parametros` repassa para o AG qualquer parâmetro nomeado. Para cada semente roda o AG, pega o último registro (`hist[-1]`) e guarda um dicionário com custo final, se achou o ótimo e a diversidade. O `_` ignora a população devolvida, porque aqui só interessa o histórico.
2. `taxa_de_sucesso`: percentual de execuções com `encontrou_otimo == True`.
3. Roda a configuração **padrão** (população 60, 100 gerações) e a **restrita** (população 8, 8 gerações).

**Por que uma configuração restrita?** Com população 60 e 100 gerações sobra recurso para um espaço de só 120 rotas, e as diferenças entre parâmetros somem. Com 8 e 8 o algoritmo "sofre" e as diferenças aparecem.

**Como ler a saída:** **96,7%** de sucesso na padrão e **43,3%** na restrita. Na tabela, a coluna de diversidade final mostra populações com 1 a 3 rotas diferentes: quase tudo virou clone.


In [21]:
# Configuração restrita para evidenciar diferenças entre parâmetros.
POP_EXP = 8
GER_EXP = 8


def repetir_execucoes(sementes, **parametros):
    '''Roda o AG para várias sementes e resume o resultado final.'''
    linhas = []
    for semente in sementes:
        _, hist = executar_algoritmo_genetico(MATRIZ, semente=semente, **parametros)
        final = hist[-1]
        linhas.append({
            "semente": semente,
            "melhor_custo": final["melhor_custo"],
            "encontrou_otimo": final["melhor_custo"] == custo_otimo,
            "diversidade_final": final["diversidade"],
        })
    return linhas


def taxa_de_sucesso(linhas):
    return 100 * sum(1 for l in linhas if l["encontrou_otimo"]) / len(linhas)


# Configuração padrão (população 60, 100 gerações).
resultados_padrao = repetir_execucoes(
    range(30),
    tamanho_populacao=TAMANHO_POPULACAO,
    numero_geracoes=NUMERO_GERACOES,
    taxa_mutacao=TAXA_MUTACAO,
)

# Configuração restrita.
resultados_restrita = repetir_execucoes(
    range(30),
    tamanho_populacao=POP_EXP,
    numero_geracoes=GER_EXP,
    taxa_mutacao=TAXA_MUTACAO,
)

print(f"Taxa de sucesso (pop {TAMANHO_POPULACAO}, {NUMERO_GERACOES} gerações): "
      f"{taxa_de_sucesso(resultados_padrao):.1f}%")
print(f"Taxa de sucesso (pop {POP_EXP}, {GER_EXP} gerações): "
      f"{taxa_de_sucesso(resultados_restrita):.1f}%")

print("\nPrimeiras 10 execuções da configuração restrita:")
imprimir_tabela(
    ("Semente", "Melhor custo", "Achou o ótimo?", "Diversidade final"),
    [(l["semente"], l["melhor_custo"], "sim" if l["encontrou_otimo"] else "não",
      l["diversidade_final"]) for l in resultados_restrita[:10]],
)

Taxa de sucesso (pop 60, 100 gerações): 96.7%
Taxa de sucesso (pop 8, 8 gerações): 43.3%

Primeiras 10 execuções da configuração restrita:
Semente | Melhor custo | Achou o ótimo? | Diversidade final
--------+--------------+----------------+------------------
0       | 150          | não            | 3                
1       | 161          | não            | 2                
2       | 204          | não            | 1                
3       | 150          | não            | 1                
4       | 161          | não            | 2                
5       | 125          | sim            | 3                
6       | 153          | não            | 1                
7       | 125          | sim            | 3                
8       | 125          | sim            | 1                
9       | 150          | não            | 1                


#### 🔎 Entendendo o código

**O que faz:** mostra **quais custos finais** apareceram nas 30 execuções da configuração restrita e quantas vezes cada um.

**Passo a passo:** um dicionário conta quantas vezes cada `melhor_custo` ocorreu, e o laço imprime em ordem crescente, com uma barra proporcional a 30 execuções.

**Como ler a saída:** o ótimo (125) apareceu **13 vezes de 30** (43,3%, o mesmo valor da célula anterior). Nas outras execuções o AG parou em rotas próximas (150, 153, 161...), isto é, **ótimos locais**.


In [22]:
# Distribuição do melhor custo final nas 30 execuções da configuração restrita.
contagem_custos = {}
for l in resultados_restrita:
    contagem_custos[l["melhor_custo"]] = contagem_custos.get(l["melhor_custo"], 0) + 1

print(f"Melhor custo final em 30 execuções (ótimo global = {custo_otimo}):")
for custo in sorted(contagem_custos):
    marcador = "  <- ótimo global" if custo == custo_otimo else ""
    print(f"{custo:4} {contagem_custos[custo]:3} {barra(contagem_custos[custo], 30)}{marcador}")

Melhor custo final em 30 execuções (ótimo global = 125):
 125  13 █████████████████  <- ótimo global
 150   8 ███████████
 153   2 ███
 161   4 █████
 186   1 █
 204   1 █
 207   1 █


## 9. Experimento: qual é o papel da mutação?

Executamos o mesmo AG (configuração restrita) com três taxas de mutação e **as mesmas sementes**, para uma comparação justa.

- taxa $0$: o cruzamento apenas reorganiza o que já existe;
- taxa moderada: introduz diversidade sem destruir constantemente os indivíduos;
- taxa alta: aumenta a exploração, mas pode desfazer boas combinações.

#### 🔎 Entendendo o código

**O que faz:** cria uma função genérica que varia **um parâmetro por vez** e resume o desempenho, e usa ela para testar a **taxa de mutação**.

**Passo a passo (`comparar_parametro`)**
1. Para cada valor da lista: copia os parâmetros fixos (`dict(fixos)`) e acrescenta o parâmetro testado (`parametros[nome] = valor`).
2. Roda o AG com **as mesmas 40 sementes** (comparação justa).
3. Calcula custo médio, taxa de sucesso e diversidade média e adiciona uma linha à tabela.

**Como ler a saída**

| Taxa de mutação | Sucesso | Diversidade média |
|---|---|---|
| 0,0 | 22,5% | 1,00 |
| 0,1 | 42,5% | 1,73 |
| 0,5 | 55,0% | 4,33 |

Com mutação 0, a diversidade final é **1,00**: todos os indivíduos viram clones e o cruzamento não cria nada novo. Quanto maior a mutação, mais diversidade e mais sucesso, pelo menos neste cenário pequeno.


In [23]:
def comparar_parametro(nome, valores, **fixos):
    '''Varia um parâmetro e resume o desempenho com as mesmas 40 sementes.'''
    linhas_tabela = []
    for valor in valores:
        parametros = dict(fixos)
        parametros[nome] = valor
        linhas = repetir_execucoes(range(40), **parametros)
        linhas_tabela.append((
            valor,
            f"{media([l['melhor_custo'] for l in linhas]):.2f}",
            f"{taxa_de_sucesso(linhas):.1f}%",
            f"{media([l['diversidade_final'] for l in linhas]):.2f}",
            barra(taxa_de_sucesso(linhas), 100, 30),
        ))
    imprimir_tabela((nome, "Custo médio", "Sucesso", "Diversidade média", "Sucesso (barra)"),
                    linhas_tabela)


comparar_parametro(
    "taxa_mutacao", [0.0, 0.10, 0.50],
    tamanho_populacao=POP_EXP, numero_geracoes=GER_EXP,
)

taxa_mutacao | Custo médio | Sucesso | Diversidade média | Sucesso (barra) 
-------------+-------------+---------+-------------------+-----------------
0.0          | 164.25      | 22.5%   | 1.00              | ███████         
0.1          | 145.55      | 42.5%   | 1.73              | █████████████   
0.5          | 139.07      | 55.0%   | 4.33              | ████████████████


## 10. Experimento: elitismo e método de seleção

Mais dois parâmetros para "brincar", ambos na configuração restrita:

- **elitismo** 0 (desligado) × 1 (ligado): o melhor custo pode piorar sem elitismo?
- **seleção** torneio × roleta: qual encontra o ótimo com mais frequência?

#### 🔎 Entendendo o código

**O que faz:** usa a mesma função para comparar **elitismo** (0 × 1) e **método de seleção** (torneio × roleta).

**Como ler a saída**
- **Elitismo:** 27,5% de sucesso sem elitismo × **42,5%** com elitismo. Guardar a melhor rota ajuda.
- **Seleção:** torneio **42,5%** × roleta **37,5%**. A roleta manteve **mais diversidade** (3,50 × 1,73), provavelmente porque os pesos `1/custo` são parecidos e ela escolhe de forma menos "exigente" que o torneio.

Lembre que são resultados de 40 execuções com uma matriz específica; servem para comparar, não para generalizar.


In [24]:
print("Elitismo:")
comparar_parametro(
    "elitismo", [0, 1],
    tamanho_populacao=POP_EXP, numero_geracoes=GER_EXP, taxa_mutacao=TAXA_MUTACAO,
)

print("\nMétodo de seleção:")
comparar_parametro(
    "metodo_selecao", ["torneio", "roleta"],
    tamanho_populacao=POP_EXP, numero_geracoes=GER_EXP, taxa_mutacao=TAXA_MUTACAO,
)

Elitismo:
elitismo | Custo médio | Sucesso | Diversidade média | Sucesso (barra)
---------+-------------+---------+-------------------+----------------
0        | 150.40      | 27.5%   | 1.82              | ████████       
1        | 145.55      | 42.5%   | 1.73              | █████████████  

Método de seleção:
metodo_selecao | Custo médio | Sucesso | Diversidade média | Sucesso (barra)
---------------+-------------+---------+-------------------+----------------
torneio        | 145.55      | 42.5%   | 1.73              | █████████████  
roleta         | 146.60      | 37.5%   | 3.50              | ███████████    


#### 🔎 Entendendo o código

**O que faz:** verifica se o **melhor custo pode piorar** de uma geração para a seguinte, com e sem elitismo.

**Passo a passo**
1. Para `elitismo = 0` e `1`, roda 40 execuções (30 gerações, mutação 0,5).
2. `melhores` é a lista do melhor custo em cada geração. O `zip(melhores, melhores[1:])` forma pares (geração anterior, geração seguinte), e o `any(depois > antes ...)` é verdadeiro se **algum** par piorou.
3. Conta em quantas execuções isso aconteceu.
4. No final imprime a curva de uma execução sem elitismo (semente 3).

**Como ler a saída:** sem elitismo, o melhor custo piorou em **36 de 40** execuções. Com elitismo, em **0 de 40**. Na curva da semente 3 dá para ver o problema: o AG chega a **125** (o ótimo), mas depois volta para **161** e só depois recupera 125. Sem elitismo, a melhor rota pode ser perdida; com elitismo, o melhor custo **nunca** sobe.


In [25]:
# O melhor custo pode PIORAR de uma geração para a seguinte?
# Contamos, em 40 execuções, quantas tiveram o melhor custo subindo em alguma geração.
for elit in [0, 1]:
    execucoes_com_piora = 0

    for semente in range(40):
        _, hist = executar_algoritmo_genetico(
            MATRIZ, tamanho_populacao=POP_EXP, numero_geracoes=30,
            taxa_mutacao=0.5, elitismo=elit, semente=semente,
        )
        melhores = [h["melhor_custo"] for h in hist]
        if any(depois > antes for antes, depois in zip(melhores, melhores[1:])):
            execucoes_com_piora += 1

    print(f"elitismo = {elit}: melhor custo piorou em {execucoes_com_piora} de 40 execuções")

# Exemplo de curva sem elitismo (semente 3).
_, hist = executar_algoritmo_genetico(
    MATRIZ, tamanho_populacao=POP_EXP, numero_geracoes=30,
    taxa_mutacao=0.5, elitismo=0, semente=3,
)
print("\nMelhor custo por geração, sem elitismo:", [h["melhor_custo"] for h in hist])

elitismo = 0: melhor custo piorou em 36 de 40 execuções


elitismo = 1: melhor custo piorou em 0 de 40 execuções

Melhor custo por geração, sem elitismo: [186, 186, 186, 186, 186, 150, 150, 150, 150, 150, 150, 150, 150, 150, 150, 125, 125, 125, 125, 125, 125, 125, 161, 125, 125, 125, 161, 161, 161, 125, 125]


# 11. Conclusões

- A representação (permutação das cidades 1 a 5) garante que toda rota gerada é válida.
- O fitness é a distância total; como é um problema de **minimização**, menor é melhor.
- Torneio e roleta favorecem rotas curtas sem tornar a escolha determinística. Na roleta, o peso $1/\text{custo}$ converte minimização em probabilidade.
- O cruzamento **OX** recombina rotas sem repetir cidades; o cruzamento de um ponto da Aula 10 não serve para permutações.
- A mutação **swap** introduz variações que o cruzamento sozinho não cria.
- O elitismo impede que a melhor rota conhecida seja perdida.
- Diversidade muito baixa indica convergência prematura.
- O resultado de uma única execução não basta para avaliar um algoritmo estocástico.
- O AG é uma **heurística**: tenta achar uma rota curta, mas não garante o ótimo. Aqui conseguimos validar porque o espaço é pequeno (120 rotas).